# Semantic Textual Similarity



### Test data

We use the five test sets from [SemEval 2012 Task 6](https://aclanthology.org/S12-1051/). Each sentence pair has a human similarity score from 0 to 5.

In [1]:
import re
import tarfile
from io import BytesIO
from pathlib import Path
from statistics import correlation
from urllib.request import urlopen

archive_paths = [
    Path('resources/test-gold.tgz'),
    Path('sts/resources/test-gold.tgz'),
    Path('test-gold.tgz'),
]
archive_path = next((path for path in archive_paths if path.is_file()), None)
if archive_path is None:
    raise FileNotFoundError('Could not find test-gold.tgz in sts/resources or the working directory.')

datasets = {
    'MSRpar': 'MSRpar',
    'MSRvid': 'MSRvid',
    'SMTeuroparl': 'SMTeuroparl',
    'OnWN': 'surprise.OnWN',
    'SMTnews': 'surprise.SMTnews',
}

def read_lines(archive, filename):
    with archive.extractfile(filename) as member:
        return member.read().decode('utf-8').splitlines()

def load_data(archive_bytes, folder, dataset_files):
    data = {}
    with tarfile.open(fileobj=BytesIO(archive_bytes), mode='r:gz') as archive:
        for name, suffix in dataset_files.items():
            pairs = read_lines(archive, f'{folder}/STS.input.{suffix}.txt')
            gold = read_lines(archive, f'{folder}/STS.gs.{suffix}.txt')
            data[name] = [
                (left, right, float(score))
                for (left, right), score in zip(
                    (pair.split('\t') for pair in pairs), gold, strict=True
                )
            ]
    return data

def print_counts(data):
    for name, rows in data.items():
        print(f'{name}: {len(rows)} pairs')
    print(f'Total: {sum(len(rows) for rows in data.values())} pairs')

test_data = load_data(archive_path.read_bytes(), 'test-gold', datasets)
print_counts(test_data)

MSRpar: 750 pairs
MSRvid: 750 pairs
SMTeuroparl: 459 pairs
OnWN: 750 pairs
SMTnews: 399 pairs
Total: 3108 pairs


### Word overlap baseline

We take the set of words in each sentence, ignoring case and punctuation. Jaccard similarity divides the number of shared words by the number of distinct words across both sentences.

In [2]:
word_pattern = re.compile(r'\w+')

def unique_sentences(data):
    return {
        sentence
        for rows in data.values()
        for left, right, _ in rows
        for sentence in (left, right)
    }

def words_by_sentence(sentences):
    return {
        sentence: {word.casefold() for word in word_pattern.findall(sentence)}
        for sentence in sentences
    }

def jaccard(first, second):
    union = first | second
    return len(first & second) / len(union) if union else 1.0

def overlap_scores(data, tokens):
    return {
        name: [jaccard(tokens[left], tokens[right]) for left, right, _ in rows]
        for name, rows in data.items()
    }

word_sets = words_by_sentence(unique_sentences(test_data))
word_scores = overlap_scores(test_data, word_sets)

### Pearson correlation

We compare the predicted scores with the human ratings for each test set and for all pairs together (ALL in the SemEval results).

In [3]:
def pearson_scores(data, scores_by_dataset):
    results = {}
    all_gold = []
    all_scores = []
    for name, rows in data.items():
        gold = [score for _, _, score in rows]
        scores = scores_by_dataset[name]
        results[name] = correlation(gold, scores)
        all_gold.extend(gold)
        all_scores.extend(scores)

    results['ALL'] = correlation(all_gold, all_scores)
    return results

for name, score in pearson_scores(test_data, word_scores).items():
    print(f'{name}: {score:.4f}')

MSRpar: 0.5529
MSRvid: 0.4449
SMTeuroparl: 0.4853
OnWN: 0.6500
SMTnews: 0.4173
ALL: 0.4773


In [4]:
example_index = 570
left, right, gold = test_data['MSRvid'][example_index]
print(left)
print(right)
print(f'Gold: {gold:.1f}/5')
print(f'Jaccard: {word_scores["MSRvid"][example_index]:.3f}')

A group of people sing.
Some people are singing.
Gold: 5.0/5
Jaccard: 0.125


Word Jaccard reaches 0.4773 on ALL, compared with 0.3110 for the word-overlap baseline. The strongest result is on OnWN (0.6500).

In the MSRvid example, the human rating is 5, but Jaccard gives 0.125 because the only shared word is PEOPLE. Word overlap misses that SING and SINGING express the same action.

### Training data

We compare new features using the three official training sets. The notebook reads a local copy of [train.tgz](https://smedina-upc.github.io/ihlt/sts/resources/train.tgz) next to the test archive if available, or downloads it from the course site.

In [5]:
train_path = archive_path.with_name('train.tgz')
if train_path.is_file():
    train_bytes = train_path.read_bytes()
else:
    train_url = 'https://smedina-upc.github.io/ihlt/sts/resources/train.tgz'
    with urlopen(train_url, timeout=30) as response:
        train_bytes = response.read()

train_datasets = {name: datasets[name] for name in ('MSRpar', 'MSRvid', 'SMTeuroparl')}
train_data = load_data(train_bytes, 'train', train_datasets)
print_counts(train_data)

MSRpar: 750 pairs
MSRvid: 750 pairs
SMTeuroparl: 734 pairs
Total: 2234 pairs


### Words and lemmas

We use NLTK to tag each word and find its base form with WordNet. Both methods use the same words and the same Jaccard calculation. The difference is whether words are reduced to lemmas. This section requires NLTK.

In [6]:
from functools import cache

import nltk
from nltk.stem import WordNetLemmatizer

resources = {
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
    'wordnet': 'corpora/wordnet/',
}
for package, resource in resources.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, quiet=True, raise_on_error=True)

token_pattern = re.compile(r'\w+|[^\w\s]')
lemmatizer = WordNetLemmatizer()
wordnet_pos = {'N': 'n', 'V': 'v', 'J': 'a', 'R': 'r'}

@cache
def lemma(word, tag):
    word = word.casefold()
    pos = wordnet_pos.get(tag[0])
    return lemmatizer.lemmatize(word, pos) if pos else word

train_sentences = sorted(unique_sentences(train_data))
tagged_sentences = nltk.pos_tag_sents(
    [token_pattern.findall(sentence) for sentence in train_sentences]
)
train_lemma_sets = {
    sentence: {lemma(word, tag) for word, tag in tagged if word_pattern.fullmatch(word)}
    for sentence, tagged in zip(train_sentences, tagged_sentences, strict=True)
}

train_word_scores = overlap_scores(train_data, words_by_sentence(train_sentences))
train_lemma_scores = overlap_scores(train_data, train_lemma_sets)

In [7]:
word_results = pearson_scores(train_data, train_word_scores)
lemma_results = pearson_scores(train_data, train_lemma_scores)

print('Dataset | Words | Lemmas')
for name in word_results:
    print(f'{name} | {word_results[name]:.4f} | {lemma_results[name]:.4f}')

Dataset | Words | Lemmas
MSRpar | 0.5637 | 0.5760
MSRvid | 0.4291 | 0.5664
SMTeuroparl | 0.4940 | 0.4991
ALL | 0.4813 | 0.5276


Lemmas improve Pearson on all three training sets. The largest gain is on MSRvid (0.4291 to 0.5664). In pair 158, grouping **grating** with **grates** and **onion** with **onions** raises Jaccard from 0.286 to 0.800. The human rating is 5.

Lemmas can also hide differences that matter. In pair 654, **woman** and **women** become the same word and the score rises from 0.100 to 0.375. The human rating is 2.